In [10]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Run from the research repository or its test/ folder.
_candidates = [Path.cwd(), *Path.cwd().parents,
               Path('/Users/kk985/PhD/Research/AllocationMultiplicity')]
ROOT = next((p for p in _candidates if (p / 'src/allocations.py').exists()), Path.cwd())
sys.path.insert(0, str(ROOT))
import src.allocations
importlib.reload(src.allocations)
from src.allocations import Allocations

In [11]:
method = "test_bootstrap_multiclass_nn"
TEST_SIZE = 5
SELECTION_RATE = 0.5
QUALIFICATION_COLUMN = "qualified_gagne_1"  

NEED_COLUMN = "gagne_sum_t"  
EXPECTED_NEED_COLUMN = "expected_need_{model}"
GROUP_COLUMN = "race"
RASHOMON_EPSILON = 0.01
ALLOCATION_DELTA = 0.1
NUM_MODELS = 100
ITERATIONS_PER_SPLIT = 25
ETA = 0.1
NEED_BOUND = 17 

fp_input = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
prediction_name = f"{method}_{QUALIFICATION_COLUMN}.csv"
_prediction_paths = [
    ROOT / "data/predictions" / prediction_name,
    ROOT / "multiclass_nn_outputs" / prediction_name,
    ROOT / "test/multiclass_nn_outputs" / prediction_name,
]
fp_predictions = next((p for p in _prediction_paths if p.exists()), _prediction_paths[0])
fp_output_opt = ROOT / f"data/results/allocations/opt_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
fp_output_lda_tru = ROOT / f"data/results/allocations/lda_tru_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
fp_output_lda_exp = ROOT / f"data/results/allocations/lda_exp_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
fp_output_people = ROOT / f"data/results/allocations/people_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"

In [12]:
data = pd.read_csv(fp_input)
df = pd.read_csv(fp_predictions)
df = df.merge(
    data[["person_id", GROUP_COLUMN]],
    on="person_id", how="left", validate="many_to_one",
)
required_need_columns = [
    Allocations.expected_need_column_for_model(f"m_{i}", EXPECTED_NEED_COLUMN)
    for i in range(1, NUM_MODELS + 1)
]
missing = [column for column in required_need_columns if column not in df]
if missing:
    raise ValueError(f"Missing model-specific expected-need columns: {missing}")

In [13]:
a = Allocations(
    df=df, TEST_SIZE=TEST_SIZE, SELECTION_RATE=SELECTION_RATE,
    QUALIFICATION_COLUMN="y", NEED_COLUMN=NEED_COLUMN, GROUP_COLUMN=GROUP_COLUMN,
    RASHOMON_EPSILON=RASHOMON_EPSILON, ALLOCATION_EPSILON=ALLOCATION_DELTA,
    NUM_MODELS=NUM_MODELS, ITERATIONS_PER_SPLIT=ITERATIONS_PER_SPLIT,
)
allocations_opt, allocations_lda_tru, allocations_lda_exp, people = a.get_allocations(expected_need_column=EXPECTED_NEED_COLUMN, eta=ETA, need_bound=NEED_BOUND,)
print(f"{len(people)} cohorts, {TEST_SIZE} people per cohort, k = {a.k}")

fp_output_opt.parent.mkdir(parents=True, exist_ok=True)
fp_output_lda_tru.parent.mkdir(parents=True, exist_ok=True)
fp_output_lda_exp.parent.mkdir(parents=True, exist_ok=True)
fp_output_people.parent.mkdir(parents=True, exist_ok=True)

allocations_opt.to_csv(fp_output_opt)
allocations_lda_tru.to_csv(fp_output_lda_tru)
allocations_lda_exp.to_csv(fp_output_lda_exp)
people.to_csv(fp_output_people)

[Split 9]: 100%|██████████| 25/25 [00:03<00:00,  7.09it/s]

250 cohorts, 5 people per cohort, k = 2
